<a href="https://colab.research.google.com/github/misrori/ai/blob/2025/youtube_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Először telepítsük a szükséges csomagokat
!pip install yt-dlp --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.9/171.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.2 MB/s eta 0:00:00


In [ ]:
# YouTube Videó/Audió Letöltő Google Colab-hoz

# Importáljuk a szükséges modulokat
import os
import yt_dlp
import zipfile
import shutil
import subprocess
from IPython.display import display, HTML, Audio, Video
from google.colab import files
import ipywidgets as widgets
from IPython.display import clear_output

def download_youtube_content(url, audio_only=False, output_dir="downloads"):
    """
    YouTube videó vagy audió letöltése.

    Args:
        url (str): A YouTube videó URL-je
        audio_only (bool): Csak audió letöltése (True) vagy videó (False)
        output_dir (str): Kimeneti könyvtár

    Returns:
        list: A letöltött fájl(ok) elérési útja
    """
    # Könyvtár létrehozása, ha nem létezik
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Fájlnév előkészítése
    output_template = os.path.join(output_dir, '%(title)s.%(ext)s')

    # Letöltési beállítások
    ydl_opts = {
        'outtmpl': output_template,
        'quiet': False,
        'no_warnings': False
    }

    output_files = []

    # Tartalom típus szerinti beállítások
    if audio_only:
        ydl_opts.update({
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
        })
        print(f"Audió letöltése a(z) {url} címről a legjobb minőségben...")

        # Audió letöltése
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            audio_file = ydl.prepare_filename(info_dict)
            audio_file = os.path.splitext(audio_file)[0] + '.mp3'
            print(f"Audió sikeresen letöltve: {audio_file}")
            output_files.append(audio_file)
    else:
        ydl_opts.update({
            'format': 'bestvideo+bestaudio/best',
            'merge_output_format': 'mp4',
        })
        print(f"Videó letöltése a(z) {url} címről a legjobb minőségben...")

        # Videó letöltése
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            video_file = ydl.prepare_filename(info_dict)
            if not video_file.endswith('.mp4'):
                video_file = os.path.splitext(video_file)[0] + '.mp4'
            print(f"Videó sikeresen letöltve: {video_file}")
            output_files.append(video_file)

    return output_files

def trim_media_file(input_file, start_time, end_time, output_dir="downloads"):
    """
    Média fájl vágása FFmpeg segítségével.

    Args:
        input_file (str): A bemeneti fájl elérési útja
        start_time (str): Kezdési idő másodpercben
        end_time (str): Befejezési idő másodpercben
        output_dir (str): Kimeneti könyvtár

    Returns:
        str: A kivágott fájl elérési útja
    """
    # Ellenőrizzük, hogy a start_time és end_time értékek konvertálhatóak-e számmá
    try:
        start_sec = float(start_time) if start_time and start_time.strip() else 0
        end_sec = float(end_time) if end_time and end_time.strip() else None
    except ValueError:
        raise ValueError("Az időpontoknak számnak kell lenniük (másodpercben)")

    # Fájlnév előkészítése
    base_name = os.path.basename(input_file)
    file_name, file_ext = os.path.splitext(base_name)

    # Trimelt fájl nevének előkészítése
    trimmed_file = os.path.join(output_dir, f"{file_name}_trimmed{file_ext}")

    # FFmpeg parancs előkészítése
    ffmpeg_cmd = ['ffmpeg', '-i', input_file, '-y']

    # Időpontok hozzáadása a parancssorhoz
    if start_sec > 0:
        ffmpeg_cmd.extend(['-ss', str(start_sec)])

    if end_sec is not None:
        # Számítsuk ki a vágás hosszát (end_sec - start_sec)
        duration = end_sec - start_sec
        ffmpeg_cmd.extend(['-t', str(duration)])

    # Kódolás beállítása
    if file_ext.lower() == '.mp3':
        ffmpeg_cmd.extend(['-acodec', 'copy'])
    elif file_ext.lower() == '.mp4':
        ffmpeg_cmd.extend(['-c', 'copy'])

    # Kimeneti fájl hozzáadása
    ffmpeg_cmd.append(trimmed_file)

    # FFmpeg parancs végrehajtása
    print(f"Média fájl vágása: {' '.join(ffmpeg_cmd)}")
    subprocess.run(ffmpeg_cmd, check=True)

    return trimmed_file

def create_zip_archive(download_dir="downloads", zip_name="youtube_downloads.zip"):
    """
    A letöltött fájlokat zip fájlba csomagolja
    """
    zip_path = os.path.join(download_dir, zip_name)
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for root, dirs, files in os.walk(download_dir):
            for file in files:
                if file != zip_name:  # Ne csomagoljuk be a zip fájlt saját magába
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, download_dir)
                    zipf.write(file_path, arcname)
    return zip_path

def download_and_process():
    """
    Videó letöltése és felhasználói felület kezelése
    """
    # Felhasználói input feldolgozása
    url = url_input.value
    audio_only = audio_only_checkbox.value
    use_time_range = time_range_checkbox.value
    auto_download = auto_download_checkbox.value

    if not url:
        status_output.value = '<p style="color: red;">Hiányzó URL! Kérlek add meg a YouTube videó URL-jét.</p>'
        return

    status_output.value = '<p>Letöltés folyamatban, kérlek várj...</p>'

    try:
        # Letöltés végrehajtása
        output_files = download_youtube_content(url, audio_only)

        # Időtartam vágása, ha szükséges
        if use_time_range:
            start_time = start_time_input.value
            end_time = end_time_input.value

            if start_time or end_time:
                trimmed_files = []
                for file in output_files:
                    try:
                        # Vágás végrehajtása
                        status_output.value = f'<p>Vágás folyamatban: {file}...</p>'
                        trimmed_file = trim_media_file(file, start_time, end_time)
                        trimmed_files.append(trimmed_file)
                    except Exception as e:
                        status_output.value = f'<p style="color: red;">Hiba a vágás során: {str(e)}</p>'
                        # Ha a vágás sikertelen, az eredeti fájlt használjuk
                        trimmed_files.append(file)

                # Sikeres vágás esetén az eredeti fájlokat felváltjuk a vágott fájlokkal
                output_files = trimmed_files

        # Sikeres letöltés jelzése
        status_output.value = '<p style="color: green;">Letöltés és feldolgozás sikeres!</p>'

        # Letöltött fájlok megjelenítése
        if output_files:
            preview_output.clear_output()
            with preview_output:
                for file in output_files:
                    if file.endswith('.mp3'):
                        print(f"Audió letöltve: {file}")
                        display(Audio(file, autoplay=False))
                    elif file.endswith('.mp4'):
                        print(f"Videó letöltve: {file}")
                        display(Video(file, width=400, height=300))

        # Automatikus letöltés, ha be van kapcsolva
        if auto_download:
            for file in output_files:
                files.download(file)

        # Zip gomb aktiválása
        download_zip_button.disabled = False

    except Exception as e:
        status_output.value = f'<p style="color: red;">Hiba történt: {str(e)}</p>'

def download_zip(_):
    """
    Letöltött fájlok mentése a felhasználó számítógépére ZIP fájlban
    """
    try:
        zip_path = create_zip_archive()
        files.download(zip_path)
        status_output.value = '<p style="color: green;">Letöltés sikeres! ZIP fájl letöltése elkezdődött.</p>'
    except Exception as e:
        status_output.value = f'<p style="color: red;">Hiba a fájlok tömörítése közben: {str(e)}</p>'

# Felhasználói felület létrehozása
url_input = widgets.Text(
    description='YouTube URL:',
    placeholder='https://www.youtube.com/watch?v=...',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

audio_only_checkbox = widgets.Checkbox(
    value=False,
    description='Csak audió letöltése',
    style={'description_width': 'initial'}
)

time_range_checkbox = widgets.Checkbox(
    value=False,
    description='Időtartam vágása',
    style={'description_width': 'initial'}
)

start_time_input = widgets.Text(
    description='Kezdési idő (mp):',
    placeholder='0',
    disabled=True,
    style={'description_width': 'initial'}
)

end_time_input = widgets.Text(
    description='Befejezési idő (mp):',
    placeholder='60',
    disabled=True,
    style={'description_width': 'initial'}
)

auto_download_checkbox = widgets.Checkbox(
    value=False,
    description='Letöltés végén lokális gépre való letöltés',
    style={'description_width': 'initial'}
)

download_button = widgets.Button(
    description='Letöltés indítása',
    button_style='primary',
    style={'description_width': 'initial'}
)

download_zip_button = widgets.Button(
    description='Fájlok mentése ZIP-ben',
    button_style='success',
    disabled=True,
    style={'description_width': 'initial'}
)

status_output = widgets.HTML(
    value='<p>Add meg a YouTube videó URL-jét és válaszd ki a letöltési beállításokat!</p>'
)

preview_output = widgets.Output()

# Időtartam opciók megjelenítése/elrejtése
def toggle_time_range(change):
    start_time_input.disabled = not change['new']
    end_time_input.disabled = not change['new']

time_range_checkbox.observe(toggle_time_range, names='value')

# Eseménykezelők hozzáadása
download_button.on_click(lambda _: download_and_process())
download_zip_button.on_click(download_zip)

# Felhasználói felület összeállítása
display(widgets.HTML(value='<h2>YouTube Videó/Audió Letöltő</h2>'))
display(url_input)
display(audio_only_checkbox)
display(time_range_checkbox)
display(widgets.HBox([start_time_input, end_time_input]))
display(auto_download_checkbox)
display(widgets.HBox([download_button, download_zip_button]))
display(status_output)
display(widgets.HTML(value='<h3>Letöltött fájlok előnézete:</h3>'))
display(preview_output)

# Létrehozzuk a downloads könyvtárat, ha még nem létezik
if not os.path.exists('downloads'):
    os.makedirs('downloads')

HTML(value='<h2>YouTube Videó/Audió Letöltő</h2>')

Text(value='', description='YouTube URL:', layout=Layout(width='80%'), placeholder='https://www.youtube.com/wa…

Checkbox(value=False, description='Csak audió letöltése', style=DescriptionStyle(description_width='initial'))

Checkbox(value=False, description='Időtartam vágása', style=DescriptionStyle(description_width='initial'))

Checkbox(value=False, description='Letöltés végén lokális gépre való letöltés', style=DescriptionStyle(descrip…

...

HTML(value='<h3>Letöltött fájlok előnézete:</h3>')

Output()

Videó letöltése a(z) https://www.youtube.com/watch?v=OF0fWKdxmKg címről a legjobb minőségben...
[youtube] Extracting URL: https://www.youtube.com/watch?v=OF0fWKdxmKg
[youtube] OF0fWKdxmKg: Downloading webpage
[youtube] OF0fWKdxmKg: Downloading tv client config
[youtube] OF0fWKdxmKg: Downloading player 82345d49
[youtube] OF0fWKdxmKg: Downloading tv player API JSON
[youtube] OF0fWKdxmKg: Downloading ios player API JSON
[youtube] OF0fWKdxmKg: Downloading m3u8 information
[info] Testing format 616
[info] OF0fWKdxmKg: Downloading 1 format(s): 616+251
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 42
[download] Destination: downloads/Magyar Péter： a Tisza nem sétál bele Orbánék csapdájába.f616.mp4
[download] 100% of   85.14MiB in 00:00:08 at 10.26MiB/s                
[download] Destination: downloads/Magyar Péter： a Tisza nem sétál bele Orbánék csapdájába.f251.webm
[download] 100% of    2.94MiB in 00:00:00 at 5.29MiB/s   
[Merger] Merging formats into "downloads/Magyar P

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>